# Scriven Testcase
This is a testcase for evaporation.  
A Vapor Bubble is growing in a superheated liquid.
Initially $t = 0$ there is no vapor in the system.  
The simulation is started from an analytical solution at $t_0 > 0$.  
We can then compare the simulated interface position in time to the analytically predicted.  
See `10.1016/j.ijheatmasstransfer.2021.121233`  

Note the following:
* A deviation from the source is the considerably larger surface tension, to deal with capillary timestep restrictions.
* How to take care of boundaries at the outflows?
* Paraview seems to not read time data correcty. Thus time annotation is missing in videos.
* Grid absolute size ($50\mu m$) is too small, then we get crude (or straight up wrong/empty) cut-cell/level-set rules. => rescale lengths to mm.
* How to initialize the level-set?  
    *   Quadratic:  + exact geometric represantation    - zero gradient at (0,0,0) not good for Stokes-Extension
    *   SDF:        + non-vanishing gradients           - inexact geometric representation; not-converging (at least for low res/amr)

In [ ]:
#r "BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [ ]:
string proj = "ScrivenProblem_v3";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

## Grid and Boundary/Initial Value configuration

Note, that in contrast to the 1-D Testcases, the interface radius is given in terms of $\alpha_A$ :  
* $R_I(t) = 2 \xi \sqrt{\alpha_A t} $

In [ ]:
static class Constants{
    public static double scale = 1000.0; // scale from 1m = 1000mm
    // careful order of declaration matters!!!
    public static double L = 187.5*1e-6 * Math.Pow(scale, 1);
    public static double Xi = 4.063;

    // material parameters
    public static double rho_A = 958.4*Math.Pow(scale, -3);
    public static double rho_B = 0.597*Math.Pow(scale, -3);

    public static double mu_A = 2.8e-4*Math.Pow(scale, -1);
    public static double mu_B = 1.26e-5*Math.Pow(scale, -1);

    public static double Sigma = 0.059 / 50000.0; // very small surface tension so that capillary timestep is bigger...

    public static double dT = 1.25; // rescale temperature to range [0, 1]
    public static double c_A = dT * 4216.0*Math.Pow(scale, 2);
    public static double c_B = dT * 2030.0*Math.Pow(scale, 2);

    public static double k_A = dT * 0.679*Math.Pow(scale, 1);
    public static double k_B = dT * 0.025*Math.Pow(scale, 1);

    public static double hVap  = 2.258e6*Math.Pow(scale, 2);
    public static double T_sat = 0.0; //373.15;
    public static double T_wall = T_sat + 1.0; //+ 1.25;

    public static double alpha_A = k_A / (c_A*rho_A);
    public static double alpha_B = k_B / (c_B*rho_B);
    public static double eps = 1.0-rho_B/rho_A;

    public static double r0 = 50*1e-6*Math.Pow(scale, 1);
    public static double t0 = Math.Pow(0.5 * r0 / Xi, 2) / alpha_A; // start time, to avoid singular massflux at t=0
    public static double tE = Math.Pow(2 * r0 / (2 * Xi),2)/alpha_A - t0; //1.5*1e-3 - t0; // Endtime, until twice the initial radius is reached
    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [ ]:
public static class BoundaryAndInitialValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfacePos(){");
            stw.WriteLine("         return (X,t) => 2 * " + Constants.Xi + " * Math.Sqrt(" + Constants.alpha_A + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfaceVel(){");
            stw.WriteLine("         return (X,t) => " + Constants.Xi + " * " + Constants.alpha_A + " / Math.Sqrt(" + Constants.alpha_A + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){"); // initialize quadratically
            stw.WriteLine("         bool signed = true;");
            stw.WriteLine("         if(signed)");
            stw.WriteLine("             return InterfacePos()(X,t) - Math.Sqrt(X[0]*X[0] + X[1]*X[1] + X[2]*X[2]);");
            stw.WriteLine("         else");
            stw.WriteLine("             return Math.Pow(InterfacePos()(X,t),2) - (X[0]*X[0] + X[1]*X[1] + X[2]*X[2]);");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.eps + " * InterfaceVel()(X,t);");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityXA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[0]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");           
            stw.WriteLine("     public static double VelocityYA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[1]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityZA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[2]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     static Func<double, double> TempInitial = r => {");
            stw.WriteLine("         return "+Constants.T_wall+"-2*Math.Pow("+Constants.Xi+",2.0)*(("+Constants.rho_B+"*("+Constants.hVap+"+("+Constants.c_A+"-"+Constants.c_B+")*("+Constants.T_wall+"-"+Constants.T_sat+")))/("+Constants.rho_A+"*"+Constants.c_A+"))*MathNet.Numerics.Integration.GaussLegendreRule.Integrate(z => Math.Exp(-Math.Pow("+Constants.Xi+",2.0)*(Math.Pow(1-z,-2.0)-2*"+Constants.eps+"*z-1)),1-"+Constants.r0+"/r,1, 15);");
            stw.WriteLine("     };");
            stw.WriteLine("     public static double TemperatureA(double[] X, double t){");
            stw.WriteLine("         double r = X.L2Norm();");
            stw.WriteLine("         double T_r = "+Constants.Xi+" / ("+Constants.k_A+" / ("+Constants.rho_B+" * "+Constants.hVap+")) * Math.Sqrt("+Constants.alpha_A+" / "+Constants.t0+");");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return TempInitial(r);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return "+Constants.T_sat+" - ("+Constants.r0+"-r) * T_r;");
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double TemperatureABoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityB(double[] X, double t){");
            stw.WriteLine("         return 0.0;");
            stw.WriteLine("     }");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }

    static public Formula Get_VAX() {
        return new Formula("BoundaryAndInitialValues.VelocityXA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_VAY() {
        return new Formula("BoundaryAndInitialValues.VelocityYA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_VAZ() {
        return new Formula("BoundaryAndInitialValues.VelocityZA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA() {
        return new Formula("BoundaryAndInitialValues.TemperatureA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureABoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode:GetPrefixCode());
    }
}

## Begin Postprocessing

In [ ]:
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Quadrature;

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.ToList();//.Where(s => s.SuccessfulTermination);
sessions = sessions.OrderBy(s => s.KeysAndQueries["id:AMR"]).ToList();
sessions

group by timestepsize

In [ ]:
var groupedsessions = sessions.GroupBy(s => s.KeysAndQueries["NoOfTimesteps"]);
groupedsessions

In [ ]:
double[] times = GenericBlas.Linspace(0.0, Constants.tE + Constants.t0, 1000);
double[] yValues = new double[times.Length];
var PhiFuncF = BoundaryAndInitialValueFactory.Get_Phi();
Func<double, double> PhiFunc = t => {
    return PhiFuncF.Evaluate(new double[3], t);
};
for(int i = 0; i < times.Length; i++){
    yValues[i] = PhiFunc(times[i] - Constants.t0);
}
Plot2Ddata RefPos = new Plot2Ddata(times, yValues, "analytical Interface Position");
RefPos.dataGroups.First().Format = new PlotFormat("k-");

#### loading timesteps, and estimating radius depending on volume

In [ ]:
public static Plot2Ddata GetRadius(IEnumerable<ISessionInfo> s, int samples = -1){
    Plot2Ddata plt = new Plot2Ddata();
    foreach(var _s in s){
        plt = plt.Merge(GetRadius(_s, samples));
    }
    return plt;
}

public static Plot2Ddata GetRadius(ISessionInfo s, int samples = -1){
    List<ISessionInfo> sess = new List<ISessionInfo>();
    sess.Add(s);
    loadsessions(s);

    void loadsessions(ISessionInfo _s){
        if(_s.RestartedFrom != Guid.Empty){
            Console.WriteLine("{0} restarted from {1}, loading as well ...", _s.ID, _s.RestartedFrom);
            ISessionInfo restart = _s.Database.Controller.DBDriver.LoadSession(_s.RestartedFrom, _s.Database);
            sess.Add(restart);
            loadsessions(restart);
        }
    }
    var ts = sess.SelectMany(_s => _s.Timesteps.Where(t => t.TimeStepNumber.Length == 1)); 
    ts = ts.OrderBy(t => t.PhysicalTime);
    Console.WriteLine("{0} has {1} timesteps in total", s.ID, ts.Count());

    int N = samples < 0 ? ts.Count() : Math.Min(ts.Count(), samples + 1);
    double[] times = new double[N + 1];
    double[] yValues = new double[times.Length];

    times[0] = ts.First().PhysicalTime + Constants.t0;
    yValues[0] = GetRadius(ts.First());
    int ind = 0;
    double inc = (double)ts.Count() /  (double)N;
    for(int i = 1; i < N; i++){
        ind = (int)Math.Floor(i * inc);
        var _ts = ts.ElementAt(ind);
        times[i] = _ts.PhysicalTime + Constants.t0;
        yValues[i] = GetRadius(_ts);
    }
    times[N] = ts.Last().PhysicalTime + Constants.t0;
    yValues[N] = GetRadius(ts.Last());

    double GetRadius(ITimestepInfo ttt){
        var phi = (LevelSet)ttt.Fields.SingleOrDefault(f => f.Identification == "Phi");
        var trk = new LevelSetTracker((GridData)phi.GridDat, BoSSS.Foundation.XDG.CutCellQuadratureMethod.Saye, 1, new[] {"A", "B"}, phi);
        trk.UpdateTracker(0.0);
        var vol = BoSSS.Solution.XNSECommon.XNSEUtils.GetSpeciesArea(trk, trk.GetSpeciesId("B"));
        // vol = 4/3 * pi * r^3 * 1/8
        // r = (6 * vol / pi)^1/3
        double r = Math.Pow(6 * vol / Math.PI, 1.0/3.0);
        return r;
    }

    var splitname = s.Name.Split('_');
    string h = splitname[2];
    string p = splitname[4];
    string l = splitname[3];
    string t = splitname[5];
    Plot2Ddata Pos = new Plot2Ddata(times, yValues, h + ", " + l + ", " + p + ", " + t );
    return Pos;
}

In [ ]:
Plot2Ddata plt = new Plot2Ddata();
foreach(var group in groupedsessions){
    plt = plt.Merge(GetRadius(group, 25));
}
// plt = plt.Merge(GetRadius(groupedsessions.ElementAt(0).First(), 25));
plt.ModFormat();
plt = plt.Merge(RefPos);
plt.ModLineColor("analytical", LineColors.Green);
plt.Xlabel = "Interface Position [mm]";
plt.Xlabel = "Time [s]";
plt.LegendAlignment = new[] {"i", "t", "l"};
plt.PlotNow()

## Growthrate

In [ ]:
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Quadrature;

In [ ]:
var sess = groupedsessions.ElementAt(1);
sess

Calculate the $L1$ relative error in the computed growthconstant:  
$E_1 = \frac{1}{t_{max}-t_0}\sum_i|\frac{x_I}{x_{I,ex}}-1|\Delta t_i$  
or as we have equally spaced timesteps:  
$E_1 = \frac{1}{N}\sum_i|\frac{x_I}{x_{I,ex}}-1|$

In [ ]:
static Formula _AnalyticalInterfacePos = BoundaryAndInitialValueFactory.Get_Phi();
static Func<double, double> AnalyticalInterfacePos = t => {
    return _AnalyticalInterfacePos.Evaluate(new double[3], t-Constants.t0);
};

public static double RelErrorGrowthConstant(Plot2Ddata.XYvalues InterfacePos){
    double err = 0.0;
    
    int N = InterfacePos.Abscissas.Length;
    for(int i = 0; i < N; i++){
        err += Math.Abs(InterfacePos.Values[i]/AnalyticalInterfacePos(InterfacePos.Abscissas[i]) - 1.0);
    }
    err = err/N;

    return err;
}

public static (string, double[], double[]) ReadReferenceCSV(string file){
    List<double> X = new List<double>();
    List<double> Y = new List<double>();
    string name;
    using(var reader = new StreamReader(file))
    {
        var line = reader.ReadLine();
        var values = line.Split(';');    
        name = "Bureš, Sato : " + values[1];

        while (!reader.EndOfStream)
        {
            line = reader.ReadLine();
            values = line.Split(';');

            X.Add(Convert.ToDouble(values[0].Replace(',','.')));
            Y.Add(Convert.ToDouble(values[1].Replace(',','.')));
        }
    }
    return (name, X.ToArray(), Y.ToArray());
}

In [ ]:
Plot2Ddata data = GetRadius(sess, -1);

In [ ]:
data.dataGroups.Last().Values.First().Display();
AnalyticalInterfacePos(data.dataGroups.First().Abscissas.First()).Display();

In [ ]:
Plot2Ddata GrowthConstantErrorRel = new Plot2Ddata();

{
    var dat = data;

    double[] grid = new double[dat.dataGroups.Length];
    double[] error = new double[dat.dataGroups.Length];

    for(int i = 0; i < dat.dataGroups.Length; i++){
        var dataGroup = dat.dataGroups[i];
        grid[i] = 1.0/Math.Pow(2.0, Convert.ToDouble(dataGroup.Name.Split(',')[1].Remove(0,4))); // normalize with base grid
        error[i] = RelErrorGrowthConstant(dataGroup) * 100.0; // in %

    }

    Plot2Ddata GrowthConstantErrorRel_p = new Plot2Ddata(grid, error, "k=2");
    GrowthConstantErrorRel      = GrowthConstantErrorRel.Merge(GrowthConstantErrorRel_p);

}
{
    var Reference = ReadReferenceCSV("BuresSato2021_Scriven_GrowthConstant.csv");
    Plot2Ddata GrowthConstantErrorRel_ref = new Plot2Ddata(Reference.Item2, Reference.Item3, Reference.Item1);
    GrowthConstantErrorRel      = GrowthConstantErrorRel.Merge(GrowthConstantErrorRel_ref);
}

var plot = GrowthConstantErrorRel;
plot.LegendAlignment = new string[] {"i", "t", "l"}; 
plot.Ylabel = "Relative error [%]";
plot.Xlabel = "1/Grid level [-]";
plot.LabelFont = 16;
plot.LabelTitleFont = 16;
plot.Title = "Relative error in growth constant";
plot.TitleFont = 24;

plot.LogX = true;
plot.XrangeMin = 0.1;
plot.XrangeMax = 2;
plot.LogX2 = true;
plot.X2rangeMin = 0.1;
plot.X2rangeMax = 2;

plot.LogY = true;
plot.YrangeMin = 0.1;
plot.YrangeMax = 20.0;

plot.ModFormat();
var gp = plot.ToGnuplot();
var p2 = gp.PlotSVG(); 

HtmlString pp = new HtmlString(
    "<div>" +
    "   <div>" +
        p2.ToString() +
    "   </div style='float:right;'>" +
    "</div>");
display(pp);

In [ ]:
plot.Regression()

In [ ]:
KeyValuePair<int, double>[] refslopes = new KeyValuePair<int, double>[] { new KeyValuePair<int, double>(2, 1.8)};
var slopes = plot.Regression().Take(1).ToDictionary(x=> Convert.ToInt32(x.Key.Split("=").ElementAt(1)),x=>x.Value);
slopes.Display();

Comparison of DOFs:  
BoSSS on finest grid (first timestep)

In [ ]:
sess.Last().GetDOF("VelocityX")+sess.Last().GetDOF("VelocityY")+sess.Last().GetDOF("Pressure")+sess.Last().GetDOF("Temperature")

Bures,Sato 2021 (estimated)

In [ ]:
Math.Pow(24, 3) * Math.Pow(4, 3) * 4

In [ ]:
Math.Pow(24, 2) * Math.Pow(4, 2) * 4

## Assertion

In [ ]:
foreach(var kvp in refslopes){
    if(kvp.Value > slopes[kvp.Key]){
        throw new ApplicationException("Slope too low for polynomial order : " + kvp.Key);
    }
}